# Quick script to join FT data from CHALKY 
## Just want GPS and time for Dorothee Bakker

Here we will write some quick loop to open all the daily f/t csv's (compiled by Brynn) and trim out the optics stuff, cut them down to ~1 minute resultion, and append them all together

In [12]:
#load dependancies
import pandas as pd
import os
from os import walk


In [24]:
# Set up directories
os.chdir('/mnt/storage/labs/mitchell/')
directory = "/mnt/storage/labs/mitchell/projects/CHALKY/dataProcessing/flowThrough/vsf/calibrated/"

In [25]:
pwd()

'/mnt/storage/labs/mitchell'

In [28]:
#test one file. Open, strip out unwanted columns, trim to every 60th line
files = os.listdir(directory)
file = files[0]

In [53]:
#define a skiprow function
n = 60
skip = lambda x: x%n !=0
#Only keep wanted columns
keep_cols = ['SysTime', 'gpstime', 'lon', 'lat', 'temp', 'Sal', 'Chl']

In [54]:
all = [] #Define blank dataset
for file in files:
    tmp = pd.read_csv(os.path.join(directory, file), skiprows = skip, usecols = keep_cols)
    all.append(tmp)
    
all_data = pd.concat(all, ignore_index = True)


,SysTime,gpstime,lon,lat,temp,Sal,Chl
0,143.654306,15.4210,-3.354370,50.043468,13.7880,35.0366,3.5490
1,143.654907,15.4302,-3.357826,50.042499,13.8490,35.0401,3.5321
2,143.655509,15.4354,-3.361294,50.041532,13.9014,35.0419,3.6166
3,143.656100,15.4445,-3.364700,50.040603,13.9503,35.0467,3.5490
4,143.656701,15.4537,-3.368183,50.039658,13.9897,35.0487,3.5828
...,...,...,...,...,...,...,...
49087,175.499965,11.5937,-12.699578,59.467107,11.9228,35.4227,0.4732
49088,175.500567,12.0034,-12.694473,59.466574,11.9241,35.4248,0.4394
49089,175.501169,12.0121,-12.690206,59.466142,11.9340,35.4271,0.4563
49090,175.501771,12.0211,-12.685726,59.465656,11.9429,35.4281,0.4563


In [59]:
#Remove outlyer points
#Calculate quantiles to define outliers
lower_lon_quantile = all_data['lon'].quantile(0.05)
upper_lon_quantile = all_data['lon'].quantile(0.95)

lower_lat_quantile = all_data['lat'].quantile(0.05)
upper_lat_quantile = all_data['lat'].quantile(0.95)
    
    #Filter data to exclude outliers
filtered_data = all_data[(all_data['lon'] >= lower_lon_quantile) &
                                (all_data['lon'] <= upper_lon_quantile) &
                        (all_data['lat'] >= lower_lat_quantile) &
                                (all_data['lat'] <= upper_lat_quantile)]


In [63]:
#Mess with the julian dates Ugh!!
year = 2024
filtered_data['datetime'] = pd.to_datetime(filtered_data['SysTime'], unit='D', origin=f'{year}-01-01')
filtered_data['date'] = filtered_data['datetime'].dt.date  # if you want just the date, without time
filtered_data

<ipython-input-63-a86a63b5d151>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['datetime'] = pd.to_datetime(filtered_data['SysTime'], unit='D', origin=f'{year}-01-01')
<ipython-input-63-a86a63b5d151>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['date'] = filtered_data['datetime'].dt.date  # if you want just the date, without time


,SysTime,gpstime,lon,lat,temp,Sal,Chl,datetime,date
39,143.677697,16.1550,-3.486843,50.007697,13.7163,34.9911,3.0758,2024-05-23 16:15:53.020800000,2024-05-23
40,143.678299,16.1642,-3.490094,50.006809,13.8070,35.0060,2.9913,2024-05-23 16:16:45.033600000,2024-05-23
41,143.678900,16.1734,-3.493344,50.005936,13.8983,35.0142,3.1096,2024-05-23 16:17:36.959999999,2024-05-23
42,143.679502,16.1826,-3.496611,50.005054,13.9597,35.0164,3.1096,2024-05-23 16:18:28.972800000,2024-05-23
43,143.680093,16.1918,-3.499859,50.004150,14.0091,35.0211,3.0251,2024-05-23 16:19:20.035199999,2024-05-23
...,...,...,...,...,...,...,...,...,...
49087,175.499965,11.5937,-12.699578,59.467107,11.9228,35.4227,0.4732,2024-06-24 11:59:56.976000000,2024-06-24
49088,175.500567,12.0034,-12.694473,59.466574,11.9241,35.4248,0.4394,2024-06-24 12:00:48.988800000,2024-06-24
49089,175.501169,12.0121,-12.690206,59.466142,11.9340,35.4271,0.4563,2024-06-24 12:01:41.001600000,2024-06-24
49090,175.501771,12.0211,-12.685726,59.465656,11.9429,35.4281,0.4563,2024-06-24 12:02:33.014400000,2024-06-24


In [64]:

filtered_data.to_csv(os.path.join(directory, "DY180_gps_minute.csv"), index = False)